# 03 – PU Assist backend API

Turns the chatbot from notebook 02 into a **web API** that the **Streamlit frontend** (a Docker container on
Data Lab / Portainer) calls. Data Lab's rule for GPU apps: *backend = notebook on Compute (GPU), frontend = container*.

```
student's browser ──► Streamlit frontend (Portainer container) ──HTTP──► this notebook (FastAPI on the H200)
                                                                          └─ same pipeline as notebook 02
```

**Run order (each time the server starts):**

| | Notebook | Why |
|---|---|---|
| 1 | `01_model_setup.ipynb` steps 1–4 | libraries + model download (the hub forgets them when the server stops) |
| 2 | `02_rag_chatbot.ipynb` (optional) | builds the search index once (~8 min) and lets you test; otherwise this notebook builds it |
| 3 | **Shut down notebook 02's kernel** | it holds ~62 GB of GPU memory; two copies would not fit |
| 4 | **this notebook → Run All** | loads 02's pipeline and starts the API; **keep this kernel running** – the API stops when it stops |

This notebook does **not** copy notebook 02's code: it runs 02's setup and pipeline cells straight from
`02_rag_chatbot.ipynb` (skipping its tests, chat box and Gradio page). Changes you make to 02 – prompts, models,
settings – are picked up here on the next run. Keep both notebooks in the same folder.

| Endpoint | What it does |
|---|---|
| `GET /health` | `{"status": "ok", ...}` – is the server up, which model, how busy |
| `POST /ask` | form fields `question`, `session_id` (optional), files `image`, `audio` (optional) → answer, sources, timings |
| `GET /map` | the campus map image – the chat page shows it under answers that used the map |
| `POST /reset` | form field `session_id` → forgets that student's chat history |

`/ask`, `/map` and `/reset` need the header `X-API-Key: <key>` (printed in step 2), so only your frontend can use the GPU.

To add documents to the knowledge base later without a restart, use **step 11** at the end.

## 1. Install the web-server libraries (once per server start)

In [ ]:
%pip install -q fastapi uvicorn python-multipart av requests

## 2. Settings and API key

The frontend container needs the **same key** (`PU_API_KEY`). A new key is generated the first time; to keep one key
for good (the hub forgets files when the server stops), paste it into `API_KEY` below.

In [ ]:
import os, secrets, socket, time

PORT = 8000                                 # the frontend calls http://<this machine>:PORT
API_KEY = ""                                # paste a fixed secret here to keep the same key after server restarts
PIPELINE_NOTEBOOK = "02_rag_chatbot.ipynb"  # the pipeline is loaded from this notebook's cells

if not API_KEY:
    if os.path.exists("api_key.txt"):
        API_KEY = open("api_key.txt").read().strip()
    else:
        API_KEY = secrets.token_urlsafe(24)
        open("api_key.txt", "w").write(API_KEY)
print("API key – set this as PU_API_KEY in the frontend container:\n ", API_KEY)

## 3. Load the chatbot pipeline from notebook 02 (~2 min; ~10 min if the index has to be built)

Runs only these cells of `02_rag_chatbot.ipynb`, in order: settings → corpus → chunk context → search models →
index → `search()` → vision-language model → helpers → Whisper → `answer_question()`.

In [ ]:
import json

PIPELINE_CELLS = [                                    # text that identifies each needed cell of notebook 02
    'DATA_DIR   = "."',                               # step 1  settings
    "corpus = [json.loads",                           # step 2  corpus
    "passages = []",                                  # step 2b chunk context
    "SentenceTransformer(EMBED_MODEL",                # step 3  search models
    "faiss.IndexFlatIP",                              # step 4  load (or build) the FAISS index
    "CLASS_LABEL = {",                                # step 5  search()
    "AutoModelForImageTextToText.from_pretrained",   # step 6  vision-language model
    "def ask_model",                                  # step 7  helpers
    "def load_audio",                                 # step 7b Whisper
    "SYSTEM_PROMPT =",                                # step 8  answer_question()
]

nb = json.load(open(PIPELINE_NOTEBOOK, encoding="utf-8"))
code_cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
t_all = time.time()
for marker in PIPELINE_CELLS:
    found = [c for c in code_cells if marker in c]
    if len(found) != 1:
        raise RuntimeError(f"Expected exactly one cell containing {marker!r} in {PIPELINE_NOTEBOOK}, found {len(found)}")
    print(f"\n▶ notebook 02 cell: {marker}")
    if not get_ipython().run_cell(found[0]).success:
        raise RuntimeError(f"Notebook 02's cell {marker!r} failed – see the error above")
print(f"\n✅ Pipeline ready in {(time.time() - t_all) / 60:.1f} min")

## 4. The API

- One question uses the GPU at a time; other students wait in a queue (`timings.queue` shows how long).
- Each student (`session_id`, created by the frontend) has their own chat history for follow-up questions;
  histories unused for 6 hours are dropped.

In [ ]:
import io, threading, uuid
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse
from PIL import Image

MAX_IMAGE_MB, MAX_AUDIO_MB, MAX_QUESTION_CHARS = 10, 25, 2000
SESSION_TTL_S = 6 * 3600

gpu_lock = threading.Lock()      # one answer at a time on the GPU
sessions = {}                    # session_id -> {"history": [(question, answer), ...], "seen": time}

def require_key(x_api_key: str | None = Header(default=None)):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing X-API-Key")

def read_upload(upload, max_mb, kind):
    if upload is None:
        return None
    data = upload.file.read()
    if len(data) > max_mb * 1024 * 1024:
        raise HTTPException(status_code=413, detail=f"{kind} is larger than {max_mb} MB")
    return data or None

app = FastAPI(title="PU Assist API", version="1.0")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "chunks": len(corpus), "busy": gpu_lock.locked(),
            "active_sessions": len(sessions)}

@app.post("/ask", dependencies=[Depends(require_key)])
def ask(question: str = Form(""), session_id: str = Form(""),
        image: UploadFile | None = File(None), audio: UploadFile | None = File(None)):
    question = question.strip()[:MAX_QUESTION_CHARS]
    image_bytes = read_upload(image, MAX_IMAGE_MB, "Image")
    audio_bytes = read_upload(audio, MAX_AUDIO_MB, "Audio")
    if not question and audio_bytes is None:
        raise HTTPException(status_code=400, detail="Send a question or a voice note.")
    img = None
    if image_bytes is not None:
        try:
            img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        except Exception:
            raise HTTPException(status_code=400, detail="Could not read the image.")
    sid = session_id or uuid.uuid4().hex

    t0 = time.time()
    with gpu_lock:
        waited = time.time() - t0
        HISTORY[:] = sessions.get(sid, {}).get("history", [])        # this student's conversation
        try:
            result = answer_question(question, img, audio_bytes)
        except Exception as e:
            if audio_bytes is not None and type(e).__module__.startswith("av"):   # PyAV could not decode it
                raise HTTPException(status_code=400, detail="Could not read the audio file.")
            if isinstance(e, ValueError):                                         # e.g. empty question
                raise HTTPException(status_code=400, detail=str(e))
            raise
        sessions[sid] = {"history": HISTORY[-MAX_HISTORY:], "seen": time.time()}

    for old in [k for k, v in sessions.items() if time.time() - v["seen"] > SESSION_TTL_S]:
        sessions.pop(old, None)

    sources = []
    for n, h in enumerate(result["hits"], 1):
        link = source_link(h)
        sources.append({"n": n, "title": source_label(h), "url": link if link.startswith("http") else None,
                        "kind": CLASS_LABEL.get(h["source_class"], h["source_class"]), "cited": n in result["cited"]})
    return {"session_id": sid, "question": result["question"],
            "heard": bool(audio_bytes) and not question, "answer": result["answer"], "sources": sources,
            "used_map": result["used_map"],
            "timings": {**{k: round(v, 1) for k, v in result["timings"].items()}, "queue": round(waited, 1)}}

@app.get("/map", dependencies=[Depends(require_key)])
def get_campus_map():                         # the chat page shows it under answers that used the map
    if not os.path.exists(CAMPUS_MAP):
        raise HTTPException(status_code=404, detail="No campus map on the server.")
    return FileResponse(CAMPUS_MAP, media_type="image/png")

@app.post("/reset", dependencies=[Depends(require_key)])
def reset(session_id: str = Form(...)):
    sessions.pop(session_id, None)
    return {"status": "cleared"}

print("API defined:", [r.path for r in app.routes if r.path in ("/health", "/ask", "/map", "/reset")])

## 5. Start the server (keeps running in the background while this kernel is alive)

In [ ]:
import requests, uvicorn

try:
    server.should_exit = True          # stop the server from a previous run of this cell
    time.sleep(1.5)
except NameError:
    pass

def port_in_use(p):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", p)) == 0

if port_in_use(PORT):
    with socket.socket() as s:
        s.bind(("", 0))
        PORT = s.getsockname()[1]
    print(f"Port busy – using port {PORT} instead (use it in the frontend's BACKEND_URL)")

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(150):
    try:
        print("✅ Server is up on port", PORT, "→", requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).json())
        break
    except requests.ConnectionError:
        time.sleep(0.2)
else:
    print("❌ Server did not start – scroll up for errors")

## 6. Test the API from inside this notebook

In [ ]:
BASE, KEY = f"http://127.0.0.1:{PORT}", {"X-API-Key": API_KEY}

r = requests.post(f"{BASE}/ask", headers=KEY, timeout=900,
                  data={"question": "What is the minimum attendance needed to write the end-term exam?"}).json()
print(r["answer"][:700], "\n")
print("cited:", [s["title"] for s in r["sources"] if s["cited"]])
print("timings:", r["timings"])

r2 = requests.post(f"{BASE}/ask", headers=KEY, timeout=900,
                   data={"question": "What if I was in hospital?", "session_id": r["session_id"]}).json()
print("\nfollow-up (same session):", r2["answer"][:400])

print("\nreset:", requests.post(f"{BASE}/reset", headers=KEY, data={"session_id": r["session_id"]}).json())
print("wrong key ->", requests.post(f"{BASE}/ask", headers={"X-API-Key": "wrong"}, data={"question": "hi"}).status_code,
      "(should be 401)")

## 7. The address the frontend uses (`BACKEND_URL`)

The frontend container must reach this notebook over Data Lab's network. These are this machine's addresses;
which one works from a Portainer container depends on Data Lab's setup (ask the Data Lab team, or test from the
container's console with `curl <url>/health`).

In [ ]:
host = socket.gethostname()
ips = sorted({a[4][0] for a in socket.getaddrinfo(host, None) if ":" not in a[4][0] and not a[4][0].startswith("127.")})
print("hostname :", host)
print("IPs      :", ", ".join(ips) or "(none found)")
print("JupyterHub user:", os.environ.get("JUPYTERHUB_USER"), "| prefix:", os.environ.get("JUPYTERHUB_SERVICE_PREFIX"))
print("\nTry these as BACKEND_URL in the frontend container:")
for url in [f"http://{ip}:{PORT}" for ip in ips] + [f"http://{host}:{PORT}"]:
    print("  ", url)
print("\nPU_API_KEY:", API_KEY)

## 8. Stop the server (only when you are done)

"Run All" also runs this cell, so it does nothing unless you set `STOP_SERVER = True` and run it on purpose.
If the server was stopped by mistake, just run step 5 again.

In [ ]:
STOP_SERVER = False          # set to True and run this cell to stop the API

if STOP_SERVER:
    server.should_exit = True
    print("Server stopped. Run step 5 to start it again.")
else:
    print(f"Server keeps running on port {PORT}. To stop it: set STOP_SERVER = True and run this cell.")

## 9. Optional: public link to the Streamlit page (Cloudflare quick tunnel)

For demos without Portainer. Runs the same Streamlit page as the Docker image **here on the hub**, next to the API,
and opens a free Cloudflare "quick tunnel" that gives a public `https://….trycloudflare.com` link.

- Anyone with the link can use the chatbot (and this GPU); traffic goes through Cloudflare – check that Data Lab
  allows it. To ask for a password first, put one in `APP_PASSWORD`.
- Questions are answered one at a time; with a whole class testing, students wait in a queue.
- The link changes every time this cell runs, and stops working when this kernel or the hub server stops.
- Stop it with: `tunnel_proc.terminate(); streamlit_proc.terminate()`

Set `MAKE_PUBLIC_LINK = True` and run this cell (it is off so that "Run All" never opens a public link by accident).

In [ ]:
import platform, re, subprocess, sys, urllib.request

MAKE_PUBLIC_LINK = False                   # set True and run this cell
APP_PASSWORD = ""                          # optional: put a password here to protect the page
FRONTEND_DIR = os.path.abspath("frontend")
RAW = "https://raw.githubusercontent.com/shreshav2404-code/itw_pu_chatbot-with-RAG-implemenattion/main/frontend/"

if MAKE_PUBLIC_LINK:
    for name in ("tunnel_proc", "streamlit_proc"):            # stop a previous run of this cell
        if name in globals() and globals()[name].poll() is None:
            globals()[name].terminate()

    # 1. the Streamlit page (same app.py as the Docker image) + Streamlit itself
    os.makedirs(os.path.join(FRONTEND_DIR, ".streamlit"), exist_ok=True)
    for f in ("app.py", ".streamlit/config.toml"):
        urllib.request.urlretrieve(RAW + f, os.path.join(FRONTEND_DIR, f))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit==1.64.0"], check=True)

    # 2. cloudflared – Cloudflare's official tunnel program (one file)
    CLOUDFLARED = os.path.expanduser("~/cloudflared")
    if not os.path.exists(CLOUDFLARED):
        arch = {"x86_64": "amd64", "aarch64": "arm64"}[platform.machine()]
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/"
                                   f"cloudflared-linux-{arch}", CLOUDFLARED)
        os.chmod(CLOUDFLARED, 0o755)

    # 3. start the page, reachable only from this machine (the tunnel is the way in)
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        UI_PORT = s.getsockname()[1]
    env = {**os.environ, "BACKEND_URL": f"http://127.0.0.1:{PORT}", "PU_API_KEY": API_KEY, "APP_PASSWORD": APP_PASSWORD}
    streamlit_proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(UI_PORT),
         "--server.address", "127.0.0.1", "--server.headless", "true"],
        cwd=FRONTEND_DIR, env=env, stdout=open(os.path.join(FRONTEND_DIR, "streamlit.log"), "w"),
        stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            requests.get(f"http://127.0.0.1:{UI_PORT}/_stcore/health", timeout=1)
            break
        except requests.ConnectionError:
            time.sleep(0.5)

    # 4. open the tunnel and print the public link
    tunnel_log = os.path.join(FRONTEND_DIR, "tunnel.log")
    tunnel_proc = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{UI_PORT}"],
                                   stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT)
    link = None
    for _ in range(60):
        time.sleep(1)
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tunnel_log).read())
        if found:
            link = found.group(0)
            break
    print("🌐 Public link:", link or f"not ready – read {tunnel_log}")
    if APP_PASSWORD:
        print("🔑 Password   :", APP_PASSWORD)
else:
    print("Public link is off. Set MAKE_PUBLIC_LINK = True and run this cell to create one.")

## 10. Optional: Gradio share link (simplest public link)

A small Gradio page (question box, photo, microphone) with a public `https://….gradio.live` link, created by
Gradio's own sharing service – no Streamlit, no Cloudflare. The link lasts about a week or until this kernel stops.
Anyone with the link can use it; set `GRADIO_PASSWORD` to require the login `student` + that password.
Each question is answered on its own (no follow-up memory), and it shares the GPU queue with the API.

Set `MAKE_GRADIO_LINK = True` and run this cell. Stop it with `gradio_demo.close()`.

In [ ]:
import gradio as gr

MAKE_GRADIO_LINK = False                   # set True and run this cell
GRADIO_PASSWORD = ""                       # optional: login "student" + this password

def gradio_answer(question, image, audio_path):
    if not (question or "").strip() and not audio_path:
        return "Please type a question or record a voice note."
    with gpu_lock:                         # one answer at a time, shared with the API
        HISTORY.clear()                    # many people share this page, so no chat memory
        return answer_question(question, image, audio_path)["markdown"]

if MAKE_GRADIO_LINK:
    try:
        gradio_demo.close()                # stop a previous run of this cell
    except NameError:
        pass
    with gr.Blocks(title="PU Assist") as gradio_demo:
        gr.Markdown("## 🎓 PU Assist – Presidency University help desk")
        with gr.Row():
            with gr.Column(scale=1):
                img_in = gr.Image(type="pil", label="Optional photo (notice, timetable)")
                audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Optional voice question")
            with gr.Column(scale=2):
                q_in = gr.Textbox(label="Your question", lines=3)
                ask_btn = gr.Button("Ask", variant="primary")
        out = gr.Markdown()
        ask_btn.click(gradio_answer, [q_in, img_in, audio_in], out)
        q_in.submit(gradio_answer, [q_in, img_in, audio_in], out)
    gradio_demo.launch(share=True, auth=("student", GRADIO_PASSWORD) if GRADIO_PASSWORD else None,
                       prevent_thread_lock=True)
else:
    print("Gradio link is off. Set MAKE_GRADIO_LINK = True and run this cell to create one.")

## 11. Optional: add new documents to the knowledge base (live – no restart, same public link)

Upload files into **`pu_bot/new_docs/`** and run this cell: PDF, Word, `.txt` / `.md` / `.csv`, `.html`, photos of
notices (`.jpg`, `.png`), audio or video (`.mp3`, `.m4a`, `.wav`, `.mp4`). It reads them (Qwen3-VL reads scanned
pages and photos, Whisper transcribes audio), embeds **only the new chunks**, adds them to the running index and
saves `pu_corpus.jsonl.gz` + the index. Files added earlier are skipped, so run it again any time.

- `SOURCE_CLASS` – label per file, e.g. `"campus_tour"` or `"user_note"` (unofficial material); files not listed are
  `"official_document"`. The tour guide is listed already, so a fee list or notice never gets the tour's label.
- `LINKS` – citation link per file, e.g. `{"fee_notice.pdf": "https://presidencyuniversity.in/..."}`
- To **update** a file, upload the edited version with the same name and run: its old chunks are replaced.
- `REMOVE` – file names to take out of the knowledge base (skipped even if they are still in `new_docs/`)

Keep one topic per file (fees in their own file, not in the tour guide) and write each fact as a full line: a table
row like `| Mechanical | 1,75,000 |` is hard to find, `B.Tech Mechanical Engineering – tuition fee 2026–27:
₹1,75,000 per year` is easy. Markdown headings (`## B.Tech fees 2026–27`) are copied into every chunk under them.

Speed: text files take seconds; each scanned page or photo ~30–60 s; students wait while it runs.
**Afterwards download `pu_corpus.jsonl.gz` to your laptop** – the hub forgets everything when the server stops.


In [ ]:
import gzip, hashlib, html, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf", "pypdfium2", "python-docx"], check=True)
from pypdf import PdfReader
import pypdfium2, docx

NEW_DOCS_DIR = "new_docs"                 # upload files here
SOURCE_CLASS = {                          # label per file; files not listed are "official_document"
    "campus_360_tour_guide.md": "campus_tour",            # "user_note" = unofficial material
}
LINKS = {                                 # citation link per file (optional)
    "campus_360_tour_guide.md": "https://presidencyuniversity.in/360-view/",
}
REMOVE = []                               # file names to take out (skipped even if still in new_docs)
CHUNK_CHARS = 1000

CLASS_LABEL.setdefault("user_note", "Added note (unofficial)")
UNOFFICIAL.add("user_note")
os.makedirs(NEW_DOCS_DIR, exist_ok=True)

def ocr(img, what):
    msgs = [{"role": "user", "content": [{"type": "image", "image": load_image(img, 1600)},
             {"type": "text", "text": f"Transcribe all readable text in this {what} exactly, keeping headings, "
                                      "lists and table rows. If there is little text, describe what it shows. "
                                      "Output only the content."}]}]
    return ask_model(msgs, max_new_tokens=1500, verbose=False)

def extract(path):
    """List of (page number or None, text)."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        pdf, pages = pypdfium2.PdfDocument(path), []
        for i, page in enumerate(PdfReader(path).pages):
            text = (page.extract_text() or "").strip()
            if len(text) < 40:                                   # scanned page: let Qwen3-VL read it
                text = ocr(pdf[i].render(scale=2).to_pil(), "page")
            pages.append((i + 1, text))
        return pages
    if ext == ".docx":
        d = docx.Document(path)
        parts = [p.text for p in d.paragraphs] + [" | ".join(c.text for c in row.cells) for t in d.tables for row in t.rows]
        return [(None, "\n".join(parts))]
    if ext in (".txt", ".md", ".csv"):
        return [(None, open(path, encoding="utf-8", errors="ignore").read())]
    if ext in (".html", ".htm"):
        raw = re.sub(r"(?is)<(script|style).*?</\1>", " ", open(path, encoding="utf-8", errors="ignore").read())
        return [(None, html.unescape(re.sub(r"<[^>]+>", "\n", raw)))]
    if ext in (".jpg", ".jpeg", ".png", ".webp", ".bmp"):
        return [(None, ocr(path, "image"))]
    if ext in (".mp3", ".m4a", ".wav", ".ogg", ".opus", ".webm", ".mp4", ".mov"):
        return [(None, transcribe(path, verbose=False))]
    return None

def split_text(text, size=CHUNK_CHARS):
    text = re.sub(r"[ \t\u00a0]+", " ", (text or "").replace("\ufffd", " ")).strip()
    chunks, cur = [], ""
    for piece in re.split(r"(?<=[.!?])\s+|\n+", text):
        piece = piece.strip()
        if not piece:
            continue
        while len(piece) > size:                                  # very long line: hard cut
            if cur:
                chunks.append(cur)
                cur = ""
            chunks.append(piece[:size])
            piece = piece[size:]
        if cur and len(cur) + 1 + len(piece) > size:
            chunks.append(cur)
            cur = piece
        else:
            cur = f"{cur} {piece}".strip()
    if cur:
        chunks.append(cur)
    return [c for c in chunks if len(c) >= 15]

def chunks_of(text):
    """split_text() per Markdown section; each chunk starts with its headings, e.g. "Fees 2026-27 › B.Tech"."""
    heads, body, out = [], [], []
    for line in (text or "").splitlines() + ["# "]:           # the last "# " flushes the final section
        m = re.match(r"(#{1,6})\s+(.*)", line)
        if not m:
            body.append(line)
            continue
        path = " › ".join(h for _, h in heads)
        out += [f"{path}\n{c}" if path else c for c in split_text("\n".join(body))]
        heads = [(lvl, h) for lvl, h in heads if lvl < len(m.group(1))] + [(len(m.group(1)), m.group(2).strip())]
        body = []
    return out

def doc_id(path):                                             # changes when the file's content changes
    with open(path, "rb") as f:
        return "add-d" + hashlib.sha1(f.read()).hexdigest()[:10]

files = {name: doc_id(os.path.join(NEW_DOCS_DIR, name)) for name in sorted(os.listdir(NEW_DOCS_DIR))
         if not name.startswith(".") and os.path.isfile(os.path.join(NEW_DOCS_DIR, name)) and name not in REMOVE}

# --- remove files in REMOVE, and the old version of every file that was edited (same name, new content)
removed = 0
drop = {i for i, r in enumerate(corpus) if r["id"].startswith("add-") and (
        r.get("local_path") in REMOVE or
        r.get("local_path") in files and not r["id"].startswith(files[r["local_path"]] + "-"))}
if drop:
    dropped = sorted({corpus[i]["local_path"] for i in drop})
    with gpu_lock:
        index.remove_ids(np.array(sorted(drop), dtype="int64"))
        corpus[:] = [r for i, r in enumerate(corpus) if i not in drop]
        passages[:] = [p for i, p in enumerate(passages) if i not in drop]
        pos_by_id.clear()
        pos_by_id.update({r["id"]: i for i, r in enumerate(corpus)})
    removed = len(drop)
    print(f"Removed {removed} old chunks of {dropped}")

# --- read new files into chunks
new = []
for name, doc in files.items():
    path = os.path.join(NEW_DOCS_DIR, name)
    if f"{doc}-0" in pos_by_id:
        print("already added:", name)
        continue
    t = time.time()
    try:
        with gpu_lock:                                            # OCR / Whisper use the GPU
            pages = extract(path)
    except Exception as e:
        print(f"could not read {name}: {type(e).__name__}: {e}")
        continue
    if pages is None:
        print("skipped (unsupported file type):", name)
        continue
    title, n, prev = os.path.splitext(name)[0].replace("_", " "), 0, None
    cls = SOURCE_CLASS.get(name, "official_document")
    for page, text in pages:
        for chunk in chunks_of(text):
            rec = {"id": f"{doc}-{n}", "text": chunk, "title": title, "source_class": cls,
                   "source_type": "added_document", "local_path": name}
            if LINKS.get(name):
                rec["url"] = LINKS[name]
            if page:
                rec["page"] = page
            if cls in UNOFFICIAL:
                rec["unverified"] = True
            tail = f"…{prev[-CONTEXT_CHARS:]} " if prev is not None else ""
            new.append((rec, f"{title}\n{tail}{chunk}".strip()))
            prev, n = chunk, n + 1
    print(f"{name}: {n} chunks ({time.time() - t:.0f}s)")

# --- embed only the new chunks and add them to the live index
if new:
    with gpu_lock:
        index.add(embed([p for _, p in new]))
        for rec, p in new:
            pos_by_id[rec["id"]] = len(corpus)
            corpus.append(rec)
            passages.append(p)

# --- save, so a kernel restart keeps the additions
if new or removed:
    with gpu_lock:
        snapshot = list(corpus)
        faiss.write_index(index, index_file)
    with gzip.open(CORPUS + ".tmp", "wt", encoding="utf-8", compresslevel=6) as f:
        for r in snapshot:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(CORPUS + ".tmp", CORPUS)
    with open(info_file, "w") as f:
        json.dump({"count": len(snapshot), "first_id": snapshot[0]["id"], "last_id": snapshot[-1]["id"],
                   "embed_model": EMBED_MODEL, "passages": f"title+prev{CONTEXT_CHARS}"}, f)
print(f"✅ Knowledge base now has {len(corpus):,} chunks (+{len(new)} added, -{removed} removed). "
      "Download pu_corpus.jsonl.gz to your laptop to keep it.")